# U-Net Spatial Baseline: Final  Colab Notebook

**Task:** Task 2 Model Building  
**Model:** U-Net Spatial Baseline  
**Target:** Next-day cumulative fire-mask prediction  

This notebook is a U-Net spatial baseline.  
It includes all required code directly in the notebook:

1. Google Drive setup  
2. Dataset path checking  
3. Normalization statistics loading/computation  
4. Normalized fire-filtered dataset class  
5. U-Net model architecture  
6. BCE + Dice loss  
7. Training loop with best/latest checkpoint saving  
8. Validation threshold sweep  
9. Fixed-threshold test evaluation  
10. Prediction visualization  

The notebook expects the `tensors_npy_cache` dataset generated from Task 1.

## 1. Mount Google Drive and Set Dataset Root

Edit only `DATA_ROOT` if the dataset path is different.

Expected dataset structure:

```text
Dataset/
├── tensors_npy_cache/
│   ├── train/
│   ├── val/
│   └── test/
└── unet_spatial_workspace/
    └── reports/
        └── module14_normalization_stats.json
```

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os

# Edit this path only if needed.
DATA_ROOT = Path("/content/drive/MyDrive/Wildfire_Area_Regression /Dataset")

os.environ["WILDFIRE_DATA_ROOT"] = str(DATA_ROOT)

print("DATA_ROOT:", DATA_ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists())

Mounted at /content/drive
DATA_ROOT: /content/drive/MyDrive/Wildfire_Area_Regression /Dataset
DATA_ROOT exists: True


## 2. Imports and Configuration

In [2]:
from pathlib import Path
import os
import json
import time
import csv
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# Core paths
# ============================================================

DATA_ROOT = Path(os.environ["WILDFIRE_DATA_ROOT"])
CACHE_ROOT = DATA_ROOT / "tensors_npy_cache"
UNET_WORKSPACE = DATA_ROOT / "unet_spatial_workspace"
STATS_PATH = UNET_WORKSPACE / "reports" / "module14_normalization_stats.json"


# ============================================================
# Model/data configuration
# ============================================================

WINDOW_SIZE = 4
CROP_SIZE = 512
INPUT_CHANNELS = 40
OUTPUT_CHANNELS = 1
BASE_CHANNELS = 16

MIN_FIRE_PIXELS = 1
SELECTED_THRESHOLD = 0.65

FULL_RUN_NAME = "unet_fireonly_normalized_firefiltered_crop512_base16_full_e10"

EPOCHS = 10
BATCH_SIZE = 1
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
BCE_WEIGHT = 0.5
DICE_WEIGHT = 0.5
POSITIVE_CROP_PROB = 0.7
NUM_WORKERS = 0


# ============================================================
# Output paths
# ============================================================

RUN_DIR = UNET_WORKSPACE / FULL_RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
LOG_DIR = RUN_DIR / "logs"
REPORT_DIR = RUN_DIR / "reports"

BEST_CHECKPOINT_PATH = CHECKPOINT_DIR / f"{FULL_RUN_NAME}_best.pt"
LATEST_CHECKPOINT_PATH = CHECKPOINT_DIR / f"{FULL_RUN_NAME}_latest.pt"

for path in [UNET_WORKSPACE, CHECKPOINT_DIR, LOG_DIR, REPORT_DIR, STATS_PATH.parent]:
    path.mkdir(parents=True, exist_ok=True)

print("CACHE_ROOT:", CACHE_ROOT)
print("UNET_WORKSPACE:", UNET_WORKSPACE)
print("STATS_PATH:", STATS_PATH)
print("RUN_DIR:", RUN_DIR)

Device: cuda
CUDA available: True
GPU: Tesla T4
CACHE_ROOT: /content/drive/MyDrive/Wildfire_Area_Regression /Dataset/tensors_npy_cache
UNET_WORKSPACE: /content/drive/MyDrive/Wildfire_Area_Regression /Dataset/unet_spatial_workspace
STATS_PATH: /content/drive/MyDrive/Wildfire_Area_Regression /Dataset/unet_spatial_workspace/reports/module14_normalization_stats.json
RUN_DIR: /content/drive/MyDrive/Wildfire_Area_Regression /Dataset/unet_spatial_workspace/unet_fireonly_normalized_firefiltered_crop512_base16_full_e10


## 3. Dataset and Path Validation

This cell confirms that the required train/validation/test cache folders are visible.

In [3]:
print("DATA_ROOT exists:", DATA_ROOT.exists())
print("CACHE_ROOT exists:", CACHE_ROOT.exists())
print("STATS_PATH exists:", STATS_PATH.exists())

for split in ["train", "val", "test"]:
    split_dir = CACHE_ROOT / split
    fire_dirs = [p for p in split_dir.iterdir() if p.is_dir()] if split_dir.exists() else []

    print("\nSplit:", split)
    print("Exists:", split_dir.exists())
    print("Fire folders:", len(fire_dirs))

    if fire_dirs:
        sample_fire = fire_dirs[0]
        print("Sample fire:", sample_fire.name)
        print("Files:", [p.name for p in sample_fire.iterdir()])

DATA_ROOT exists: True
CACHE_ROOT exists: True
STATS_PATH exists: True

Split: train
Exists: True
Fire folders: 93
Sample fire: 20700973
Files: ['fire_masks.npy', 'weather.npy', 'terrain.npy']

Split: val
Exists: True
Fire folders: 21
Sample fire: 20562846
Files: ['fire_masks.npy', 'weather.npy', 'terrain.npy']

Split: test
Exists: True
Fire folders: 21
Sample fire: 20777508
Files: ['fire_masks.npy', 'weather.npy', 'terrain.npy']


## 4. Normalization Statistics

Weather and terrain channels are normalized using train-set statistics.  
If `module14_normalization_stats.json` already exists, it is loaded directly.  
If it does not exist, this notebook can compute it from `tensors_npy_cache/train`.

In [4]:
WEATHER_VAR_NAMES = [
    "temperature_2m",
    "relative_humidity_pct",
    "wind_speed_ms",
    "wind_dir_deg",
    "surface_pressure",
    "total_precipitation",
]

TERRAIN_BAND_NAMES = [
    "elevation_m",
    "slope_deg",
    "aspect_deg",
]


def _update_stats_from_array(arr, sums, sq_sums, counts, channel_idx):
    arr = np.asarray(arr, dtype=np.float64)
    mask = np.isfinite(arr)

    if not mask.any():
        return

    valid = arr[mask]
    sums[channel_idx] += valid.sum()
    sq_sums[channel_idx] += np.square(valid).sum()
    counts[channel_idx] += valid.size


def compute_normalization_stats(cache_root: Path, output_json: Path, split: str = "train"):
    split_dir = cache_root / split

    if not split_dir.exists():
        raise FileNotFoundError(f"Split folder not found: {split_dir}")

    fire_dirs = sorted([p for p in split_dir.iterdir() if p.is_dir()])

    if not fire_dirs:
        raise RuntimeError(f"No fire folders found in: {split_dir}")

    weather_sum = np.zeros(6, dtype=np.float64)
    weather_sq_sum = np.zeros(6, dtype=np.float64)
    weather_count = np.zeros(6, dtype=np.float64)

    terrain_sum = np.zeros(3, dtype=np.float64)
    terrain_sq_sum = np.zeros(3, dtype=np.float64)
    terrain_count = np.zeros(3, dtype=np.float64)

    for fire_dir in tqdm(fire_dirs, desc=f"Computing stats from {split}"):
        weather_path = fire_dir / "weather.npy"
        terrain_path = fire_dir / "terrain.npy"

        if not weather_path.exists() or not terrain_path.exists():
            continue

        weather = np.load(weather_path, mmap_mode="r")
        terrain = np.load(terrain_path, mmap_mode="r")

        for day in range(weather.shape[0]):
            for c in range(6):
                _update_stats_from_array(
                    arr=weather[day, c],
                    sums=weather_sum,
                    sq_sums=weather_sq_sum,
                    counts=weather_count,
                    channel_idx=c,
                )

        for c in range(3):
            _update_stats_from_array(
                arr=terrain[c],
                sums=terrain_sum,
                sq_sums=terrain_sq_sum,
                counts=terrain_count,
                channel_idx=c,
            )

    weather_mean = weather_sum / np.maximum(weather_count, 1)
    weather_var = weather_sq_sum / np.maximum(weather_count, 1) - np.square(weather_mean)
    weather_std = np.sqrt(np.maximum(weather_var, 1e-12))

    terrain_mean = terrain_sum / np.maximum(terrain_count, 1)
    terrain_var = terrain_sq_sum / np.maximum(terrain_count, 1) - np.square(terrain_mean)
    terrain_std = np.sqrt(np.maximum(terrain_var, 1e-12))

    stats = {
        "split": split,
        "weather_var_names": WEATHER_VAR_NAMES,
        "terrain_band_names": TERRAIN_BAND_NAMES,
        "weather_mean": weather_mean.tolist(),
        "weather_std": weather_std.tolist(),
        "terrain_mean": terrain_mean.tolist(),
        "terrain_std": terrain_std.tolist(),
        "weather_count": weather_count.tolist(),
        "terrain_count": terrain_count.tolist(),
    }

    output_json.parent.mkdir(parents=True, exist_ok=True)

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2)

    return stats


if STATS_PATH.exists():
    print("Normalization stats already exist:")
    print(STATS_PATH)
    with open(STATS_PATH, "r", encoding="utf-8") as f:
        stats_preview = json.load(f)
else:
    print("Normalization stats not found. Computing from train split.")
    stats_preview = compute_normalization_stats(CACHE_ROOT, STATS_PATH, split="train")

print("\nWeather stats:")
for name, mean, std in zip(
    stats_preview["weather_var_names"],
    stats_preview["weather_mean"],
    stats_preview["weather_std"],
):
    print(f"{name:25s} mean={mean:.4f}, std={std:.4f}")

print("\nTerrain stats:")
for name, mean, std in zip(
    stats_preview["terrain_band_names"],
    stats_preview["terrain_mean"],
    stats_preview["terrain_std"],
):
    print(f"{name:25s} mean={mean:.4f}, std={std:.4f}")

Normalization stats already exist:
/content/drive/MyDrive/Wildfire_Area_Regression /Dataset/unet_spatial_workspace/reports/module14_normalization_stats.json

Weather stats:
temperature_2m            mean=294.9458, std=8.7636
relative_humidity_pct     mean=34.0303, std=16.9800
wind_speed_ms             mean=1.6174, std=1.1681
wind_dir_deg              mean=0.5493, std=125.6277
surface_pressure          mean=88288.1876, std=7322.0374
total_precipitation       mean=0.0005, std=0.0021

Terrain stats:
elevation_m               mean=1204.8650, std=740.8186
slope_deg                 mean=11.8420, std=9.8212
aspect_deg                mean=166.6833, std=94.2062


## 5. Dataset Class: Normalized Fire-Filtered U-Net Dataset

The dataset:

- creates four-day temporal windows,
- filters out empty windows where both current and target fire are empty,
- uses fire-aware 512×512 crops,
- normalizes weather and terrain channels,
- keeps fire masks as binary 0/1.

In [5]:
class WildfireUNetNormalizedFireFilteredDataset(Dataset):
    def __init__(
        self,
        cache_root,
        split,
        normalization_stats_path,
        window_size=4,
        crop_size=512,
        crop_mode="fire_aware",
        positive_crop_prob=0.7,
        require_fire=True,
        min_fire_pixels=1,
        limit_samples=None,
        fill_nan_value=0.0,
        eps=1e-6,
    ):
        self.cache_root = Path(cache_root)
        self.split = split
        self.split_dir = self.cache_root / split
        self.window_size = window_size
        self.crop_size = crop_size
        self.crop_mode = crop_mode
        self.positive_crop_prob = positive_crop_prob
        self.require_fire = require_fire
        self.min_fire_pixels = min_fire_pixels
        self.fill_nan_value = fill_nan_value
        self.eps = eps

        if split not in {"train", "val", "test"}:
            raise ValueError(f"Invalid split: {split}")

        if crop_mode not in {"random", "center", "fire_aware"}:
            raise ValueError("crop_mode must be random, center, or fire_aware")

        if not self.split_dir.exists():
            raise FileNotFoundError(f"Split folder not found: {self.split_dir}")

        normalization_stats_path = Path(normalization_stats_path)

        if not normalization_stats_path.exists():
            raise FileNotFoundError(f"Normalization stats not found: {normalization_stats_path}")

        with open(normalization_stats_path, "r", encoding="utf-8") as f:
            stats = json.load(f)

        self.weather_mean = np.array(stats["weather_mean"], dtype=np.float32)
        self.weather_std = np.maximum(np.array(stats["weather_std"], dtype=np.float32), eps)

        self.terrain_mean = np.array(stats["terrain_mean"], dtype=np.float32)
        self.terrain_std = np.maximum(np.array(stats["terrain_std"], dtype=np.float32), eps)

        self.fire_dirs = sorted([p for p in self.split_dir.iterdir() if p.is_dir()])
        self.samples = []
        self._mmap_cache = {}

        self._build_sample_index()

        if limit_samples is not None:
            self.samples = self.samples[:limit_samples]

    def _build_sample_index(self):
        for fire_dir in self.fire_dirs:
            fire_id = fire_dir.name

            fire_masks_path = fire_dir / "fire_masks.npy"
            weather_path = fire_dir / "weather.npy"
            terrain_path = fire_dir / "terrain.npy"

            if not (fire_masks_path.exists() and weather_path.exists() and terrain_path.exists()):
                continue

            fire_masks = np.load(fire_masks_path, mmap_mode="r")

            n_days = fire_masks.shape[0]
            n_samples = n_days - self.window_size

            if n_samples <= 0:
                continue

            for start_day in range(n_samples):
                current_day = start_day + self.window_size - 1
                target_day = start_day + self.window_size

                current_mask = fire_masks[current_day] > 0
                target_mask = fire_masks[target_day] > 0

                union_pixels = int((current_mask | target_mask).sum())

                if self.require_fire and union_pixels < self.min_fire_pixels:
                    continue

                self.samples.append({
                    "fire_id": fire_id,
                    "fire_dir": fire_dir,
                    "start_day": start_day,
                    "union_fire_pixels": union_pixels,
                })

    def __len__(self):
        return len(self.samples)

    def _load_arrays(self, fire_id, fire_dir):
        if fire_id not in self._mmap_cache:
            self._mmap_cache[fire_id] = {
                "fire_masks": np.load(fire_dir / "fire_masks.npy", mmap_mode="r"),
                "weather": np.load(fire_dir / "weather.npy", mmap_mode="r"),
                "terrain": np.load(fire_dir / "terrain.npy", mmap_mode="r"),
            }
        return self._mmap_cache[fire_id]

    def _safe_float_array(self, arr):
        arr = arr.astype(np.float32, copy=False)
        arr = np.nan_to_num(
            arr,
            nan=self.fill_nan_value,
            posinf=self.fill_nan_value,
            neginf=self.fill_nan_value,
        )
        return arr

    def _normalize_weather(self, weather):
        weather = self._safe_float_array(weather)
        mean = self.weather_mean[None, :, None, None]
        std = self.weather_std[None, :, None, None]
        return ((weather - mean) / std).astype(np.float32)

    def _normalize_terrain(self, terrain):
        terrain = self._safe_float_array(terrain)
        mean = self.terrain_mean[:, None, None]
        std = self.terrain_std[:, None, None]
        return ((terrain - mean) / std).astype(np.float32)

    def _center_crop(self, height, width):
        crop = self.crop_size
        top = (height - crop) // 2
        left = (width - crop) // 2
        return top, left, top + crop, left + crop

    def _random_crop(self, height, width):
        crop = self.crop_size
        top = np.random.randint(0, height - crop + 1)
        left = np.random.randint(0, width - crop + 1)
        return top, left, top + crop, left + crop

    def _crop_from_center(self, center_y, center_x, height, width):
        crop = self.crop_size
        top = int(center_y - crop // 2)
        left = int(center_x - crop // 2)

        top = max(0, min(top, height - crop))
        left = max(0, min(left, width - crop))

        return top, left, top + crop, left + crop

    def _fire_aware_crop(self, fire_masks, start_day, height, width):
        current_day = start_day + self.window_size - 1
        target_day = start_day + self.window_size

        fire_union = (fire_masks[current_day] > 0) | (fire_masks[target_day] > 0)
        ys, xs = np.where(fire_union)

        if len(ys) == 0:
            if self.split == "train":
                top, left, bottom, right = self._random_crop(height, width)
                return top, left, bottom, right, "fallback_random_no_fire"
            top, left, bottom, right = self._center_crop(height, width)
            return top, left, bottom, right, "fallback_center_no_fire"

        if self.split == "train":
            use_positive = np.random.rand() < self.positive_crop_prob

            if not use_positive:
                top, left, bottom, right = self._random_crop(height, width)
                return top, left, bottom, right, "random_background_mixed"

            idx = np.random.randint(0, len(ys))
            center_y = int(ys[idx])
            center_x = int(xs[idx])
        else:
            center_y = int(np.median(ys))
            center_x = int(np.median(xs))

        top, left, bottom, right = self._crop_from_center(center_y, center_x, height, width)

        source = "fire_aware_random" if self.split == "train" else "fire_aware_center"
        return top, left, bottom, right, source

    def _get_crop_coords(self, fire_masks, start_day, height, width):
        if self.crop_size is None:
            return 0, 0, height, width, "full"

        if height < self.crop_size or width < self.crop_size:
            raise ValueError(f"Image smaller than crop size: {(height, width)} < {self.crop_size}")

        if self.crop_mode == "center":
            top, left, bottom, right = self._center_crop(height, width)
            return top, left, bottom, right, "center"

        if self.crop_mode == "random":
            top, left, bottom, right = self._random_crop(height, width)
            return top, left, bottom, right, "random"

        return self._fire_aware_crop(fire_masks, start_day, height, width)

    def __getitem__(self, idx):
        item = self.samples[idx]

        fire_id = item["fire_id"]
        fire_dir = item["fire_dir"]
        start_day = item["start_day"]

        arrays = self._load_arrays(fire_id, fire_dir)

        fire_masks = arrays["fire_masks"]
        weather = arrays["weather"]
        terrain = arrays["terrain"]

        _, height, width = fire_masks.shape

        top, left, bottom, right, crop_source = self._get_crop_coords(
            fire_masks=fire_masks,
            start_day=start_day,
            height=height,
            width=width,
        )

        W = self.window_size
        i = start_day

        x_fire = fire_masks[i:i + W, top:bottom, left:right].astype(np.float32)
        x_fire = x_fire[:, None, :, :]

        x_weather = weather[i:i + W, :, top:bottom, left:right]
        x_weather = self._normalize_weather(x_weather)

        x_terrain = terrain[:, top:bottom, left:right]
        x_terrain = self._normalize_terrain(x_terrain)
        x_terrain = np.repeat(x_terrain[None, :, :, :], repeats=W, axis=0)

        x_temporal = np.concatenate([x_fire, x_weather, x_terrain], axis=1)
        T, C, H, Wd = x_temporal.shape
        x_unet = x_temporal.reshape(T * C, H, Wd)

        y_fire = fire_masks[i + self.window_size, top:bottom, left:right].astype(np.float32)
        y_fire = y_fire[None, :, :]

        return {
            "x": torch.from_numpy(x_unet).float(),
            "y_fire": torch.from_numpy(y_fire).float(),
            "fire_id": fire_id,
            "split": self.split,
            "start_day": start_day,
            "crop": {
                "top": top,
                "left": left,
                "bottom": bottom,
                "right": right,
                "source": crop_source,
            },
        }


def describe_sample(sample):
    x = sample["x"]
    y_fire = sample["y_fire"]

    print("fire_id:", sample["fire_id"])
    print("split:", sample["split"])
    print("start_day:", sample["start_day"])
    print("crop:", sample["crop"])
    print("x shape:", tuple(x.shape))
    print("y_fire shape:", tuple(y_fire.shape))
    print("x dtype:", x.dtype)
    print("y_fire dtype:", y_fire.dtype)
    print("x NaN count:", torch.isnan(x).sum().item())
    print("target pixels:", int((y_fire[0] > 0.5).sum().item()))

## 6. Dataset Sanity Check

In [ ]:
train_check_ds = WildfireUNetNormalizedFireFilteredDataset(
    cache_root=CACHE_ROOT,
    split="train",
    normalization_stats_path=STATS_PATH,
    window_size=WINDOW_SIZE,
    crop_size=CROP_SIZE,
    crop_mode="fire_aware",
    positive_crop_prob=POSITIVE_CROP_PROB,
    require_fire=True,
    min_fire_pixels=MIN_FIRE_PIXELS,
    limit_samples=5,
)

val_check_ds = WildfireUNetNormalizedFireFilteredDataset(
    cache_root=CACHE_ROOT,
    split="val",
    normalization_stats_path=STATS_PATH,
    window_size=WINDOW_SIZE,
    crop_size=CROP_SIZE,
    crop_mode="fire_aware",
    positive_crop_prob=1.0,
    require_fire=True,
    min_fire_pixels=MIN_FIRE_PIXELS,
    limit_samples=5,
)

print("Train check samples:", len(train_check_ds))
print("Val check samples:", len(val_check_ds))

sample = train_check_ds[0]
describe_sample(sample)

## 7. U-Net Model Definition

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.MaxPool2d(kernel_size=2, stride=2),
            DoubleConv(in_channels, out_channels),
        )

    def forward(self, x):
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2,
        )

        self.conv = DoubleConv(
            in_channels=out_channels + skip_channels,
            out_channels=out_channels,
        )

    def forward(self, x, skip):
        x = self.up(x)

        if x.shape[-2:] != skip.shape[-2:]:
            diff_h = skip.shape[-2] - x.shape[-2]
            diff_w = skip.shape[-1] - x.shape[-1]

            skip = skip[
                :,
                :,
                diff_h // 2: skip.shape[-2] - (diff_h - diff_h // 2),
                diff_w // 2: skip.shape[-1] - (diff_w - diff_w // 2),
            ]

        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


class UNetFire(nn.Module):
    def __init__(self, in_channels=40, base_channels=16, out_channels=1):
        super().__init__()

        b = base_channels

        self.enc1 = DoubleConv(in_channels, b)
        self.enc2 = DownBlock(b, b * 2)
        self.enc3 = DownBlock(b * 2, b * 4)
        self.enc4 = DownBlock(b * 4, b * 8)

        self.bottleneck = DownBlock(b * 8, b * 16)

        self.up4 = UpBlock(b * 16, b * 8, b * 8)
        self.up3 = UpBlock(b * 8, b * 4, b * 4)
        self.up2 = UpBlock(b * 4, b * 2, b * 2)
        self.up1 = UpBlock(b * 2, b, b)

        self.fire_head = nn.Conv2d(b, out_channels, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)

        b = self.bottleneck(s4)

        x = self.up4(b, s4)
        x = self.up3(x, s3)
        x = self.up2(x, s2)
        x = self.up1(x, s1)

        return self.fire_head(x)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


model_test = UNetFire(INPUT_CHANNELS, BASE_CHANNELS, OUTPUT_CHANNELS)
print("Trainable parameters:", count_parameters(model_test))

x_test = torch.randn(1, INPUT_CHANNELS, CROP_SIZE, CROP_SIZE)
with torch.no_grad():
    y_test = model_test(x_test)

print("Input shape:", x_test.shape)
print("Output shape:", y_test.shape)
del model_test, x_test, y_test

## 8. Loss and Metric Functions

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)

        probs = probs.view(probs.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        intersection = (probs * targets).sum(dim=1)
        union = probs.sum(dim=1) + targets.sum(dim=1)

        dice_score = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return (1.0 - dice_score).mean()


class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        return self.bce_weight * self.bce(logits, targets) + self.dice_weight * self.dice(logits, targets)


def binary_segmentation_counts(logits, targets, threshold=0.65):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()
    targets = (targets >= 0.5).float()

    tp = ((preds == 1) & (targets == 1)).sum().item()
    fp = ((preds == 1) & (targets == 0)).sum().item()
    fn = ((preds == 0) & (targets == 1)).sum().item()
    tn = ((preds == 0) & (targets == 0)).sum().item()

    return {"tp": float(tp), "fp": float(fp), "fn": float(fn), "tn": float(tn)}


def merge_counts(total, new):
    for key in ["tp", "fp", "fn", "tn"]:
        total[key] = total.get(key, 0.0) + new.get(key, 0.0)
    return total


def metrics_from_counts(counts, eps=1e-7):
    tp = counts["tp"]
    fp = counts["fp"]
    fn = counts["fn"]
    tn = counts["tn"]

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    iou = tp / (tp + fp + fn + eps)
    f1 = 2.0 * precision * recall / (precision + recall + eps)
    accuracy = (tp + tn) / (tp + fp + fn + tn + eps)

    return {
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "accuracy": float(accuracy),
    }

## 9. Create Full Train and Validation Datasets

In [ ]:
full_train_ds = WildfireUNetNormalizedFireFilteredDataset(
    cache_root=CACHE_ROOT,
    split="train",
    normalization_stats_path=STATS_PATH,
    window_size=WINDOW_SIZE,
    crop_size=CROP_SIZE,
    crop_mode="fire_aware",
    positive_crop_prob=POSITIVE_CROP_PROB,
    require_fire=True,
    min_fire_pixels=MIN_FIRE_PIXELS,
    limit_samples=None,
)

full_val_ds = WildfireUNetNormalizedFireFilteredDataset(
    cache_root=CACHE_ROOT,
    split="val",
    normalization_stats_path=STATS_PATH,
    window_size=WINDOW_SIZE,
    crop_size=CROP_SIZE,
    crop_mode="fire_aware",
    positive_crop_prob=1.0,
    require_fire=True,
    min_fire_pixels=MIN_FIRE_PIXELS,
    limit_samples=None,
)

full_train_loader = DataLoader(
    full_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

full_val_loader = DataLoader(
    full_val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Full train samples:", len(full_train_ds))
print("Full val samples:", len(full_val_ds))

sample = full_train_ds[0]
describe_sample(sample)

## 10. Training and Validation Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, threshold=0.65, print_every=100):
    model.train()

    total_loss = 0.0
    total_counts = {"tp": 0.0, "fp": 0.0, "fn": 0.0, "tn": 0.0}

    for batch_idx, batch in enumerate(tqdm(loader, desc="Training", leave=False), start=1):
        x = batch["x"].to(device, non_blocking=True)
        y_fire = batch["y_fire"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y_fire)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        counts = binary_segmentation_counts(logits.detach(), y_fire.detach(), threshold=threshold)
        total_counts = merge_counts(total_counts, counts)

        if batch_idx % print_every == 0:
            print(f"  Batch {batch_idx}/{len(loader)} | Loss: {loss.item():.6f}")

    avg_loss = total_loss / max(len(loader), 1)
    metrics = metrics_from_counts(total_counts)

    return avg_loss, metrics, total_counts


@torch.no_grad()
def evaluate_one_epoch(model, loader, criterion, device, threshold=0.65):
    model.eval()

    total_loss = 0.0
    total_counts = {"tp": 0.0, "fp": 0.0, "fn": 0.0, "tn": 0.0}

    for batch in tqdm(loader, desc="Validation", leave=False):
        x = batch["x"].to(device, non_blocking=True)
        y_fire = batch["y_fire"].to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y_fire)

        total_loss += loss.item()

        counts = binary_segmentation_counts(logits.detach(), y_fire.detach(), threshold=threshold)
        total_counts = merge_counts(total_counts, counts)

    avg_loss = total_loss / max(len(loader), 1)
    metrics = metrics_from_counts(total_counts)

    return avg_loss, metrics, total_counts


def save_checkpoint(path, model, optimizer, scheduler, epoch, train_metrics, val_metrics, train_loss, val_loss):
    path.parent.mkdir(parents=True, exist_ok=True)

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "train_metrics": train_metrics,
            "val_metrics": val_metrics,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "run_name": FULL_RUN_NAME,
            "config": {
                "window_size": WINDOW_SIZE,
                "crop_size": CROP_SIZE,
                "input_channels": INPUT_CHANNELS,
                "base_channels": BASE_CHANNELS,
                "epochs": EPOCHS,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "threshold": SELECTED_THRESHOLD,
                "loss": "BCE + Dice",
                "evaluation_type": "filtered fire-aware patch-level evaluation",
            },
        },
        path,
    )

## 11. Full Training

This cell trains the final U-Net model on all filtered train samples and validates on all filtered validation samples.

Outputs saved:

```text
checkpoints/latest.pt
checkpoints/best.pt
logs/full_training_log.csv
training_config.json
```

In [ ]:
if device.type != "cuda":
    raise RuntimeError("CUDA GPU is required for full training. Set Colab runtime to T4 GPU.")

torch.cuda.empty_cache()

full_model = UNetFire(
    in_channels=INPUT_CHANNELS,
    base_channels=BASE_CHANNELS,
    out_channels=OUTPUT_CHANNELS,
).to(device)

criterion = BCEDiceLoss(
    bce_weight=BCE_WEIGHT,
    dice_weight=DICE_WEIGHT,
)

optimizer = optim.AdamW(
    full_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6,
)

training_config = {
    "run_name": FULL_RUN_NAME,
    "train_samples": len(full_train_ds),
    "val_samples": len(full_val_ds),
    "window_size": WINDOW_SIZE,
    "crop_size": CROP_SIZE,
    "input_channels": INPUT_CHANNELS,
    "base_channels": BASE_CHANNELS,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "threshold_for_epoch_metrics": SELECTED_THRESHOLD,
    "normalization_stats": str(STATS_PATH),
}

with open(RUN_DIR / "training_config.json", "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=2)

print("Model parameters:", count_parameters(full_model))
print("Run dir:", RUN_DIR)
print("Best checkpoint:", BEST_CHECKPOINT_PATH)
print("Latest checkpoint:", LATEST_CHECKPOINT_PATH)

### 11.1 Run Training Loop

This cell saves only the latest and best checkpoints to avoid unnecessary storage use.

In [ ]:
best_val_loss = float("inf")
best_epoch = None
training_rows = []

start_time_all = time.time()

for epoch in range(1, EPOCHS + 1):
    print("\n" + "=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print("=" * 80)

    epoch_start = time.time()

    train_loss, train_metrics, train_counts = train_one_epoch(
        model=full_model,
        loader=full_train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        threshold=SELECTED_THRESHOLD,
        print_every=100,
    )

    val_loss, val_metrics, val_counts = evaluate_one_epoch(
        model=full_model,
        loader=full_val_loader,
        criterion=criterion,
        device=device,
        threshold=SELECTED_THRESHOLD,
    )

    scheduler.step()

    epoch_time_min = (time.time() - epoch_start) / 60.0
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Train Loss: {train_loss:.6f} | "
        f"Train IoU: {train_metrics['iou']:.4f} | "
        f"Train F1: {train_metrics['f1']:.4f}"
    )

    print(
        f"Val Loss: {val_loss:.6f} | "
        f"Val IoU: {val_metrics['iou']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"Val Precision: {val_metrics['precision']:.4f} | "
        f"Val Recall: {val_metrics['recall']:.4f}"
    )

    print(f"Learning rate: {current_lr:.8f}")
    print(f"Epoch time: {epoch_time_min:.2f} minutes")

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_iou": train_metrics["iou"],
        "train_precision": train_metrics["precision"],
        "train_recall": train_metrics["recall"],
        "train_f1": train_metrics["f1"],
        "train_accuracy": train_metrics["accuracy"],
        "val_iou": val_metrics["iou"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "val_accuracy": val_metrics["accuracy"],
        "lr": current_lr,
        "epoch_time_min": epoch_time_min,
    }

    training_rows.append(row)

    save_checkpoint(
        path=LATEST_CHECKPOINT_PATH,
        model=full_model,
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=epoch,
        train_metrics=train_metrics,
        val_metrics=val_metrics,
        train_loss=train_loss,
        val_loss=val_loss,
    )
    print("Saved latest checkpoint:", LATEST_CHECKPOINT_PATH)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch

        save_checkpoint(
            path=BEST_CHECKPOINT_PATH,
            model=full_model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            train_metrics=train_metrics,
            val_metrics=val_metrics,
            train_loss=train_loss,
            val_loss=val_loss,
        )
        print("Saved BEST checkpoint:", BEST_CHECKPOINT_PATH)

    df_training_log = pd.DataFrame(training_rows)
    training_log_csv = LOG_DIR / "full_training_log.csv"
    df_training_log.to_csv(training_log_csv, index=False)
    print("Updated training log:", training_log_csv)

total_time_min = (time.time() - start_time_all) / 60.0

print("\n" + "=" * 80)
print("Full training complete.")
print("=" * 80)
print("Best epoch:", best_epoch)
print("Best val loss:", best_val_loss)
print("Total training time:", total_time_min, "minutes")
print("Training log:", training_log_csv)
print("Checkpoints:", CHECKPOINT_DIR)

## 12. Load Selected Checkpoint

This cell loads the best checkpoint if available; otherwise it uses the latest checkpoint.

In [ ]:
CHECKPOINT_PATH = BEST_CHECKPOINT_PATH if BEST_CHECKPOINT_PATH.exists() else LATEST_CHECKPOINT_PATH

print("Selected checkpoint:", CHECKPOINT_PATH)
print("Exists:", CHECKPOINT_PATH.exists())

eval_model = UNetFire(
    in_channels=INPUT_CHANNELS,
    base_channels=BASE_CHANNELS,
    out_channels=OUTPUT_CHANNELS,
).to(device)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
eval_model.load_state_dict(checkpoint["model_state_dict"])
eval_model.eval()

print("Loaded checkpoint epoch:", checkpoint.get("epoch"))
print("Run name:", checkpoint.get("run_name"))

## 13. Validation Threshold Sweep

Threshold selection is done on validation data only.

In [ ]:
@torch.no_grad()
def evaluate_loader_at_threshold(model, loader, threshold, device, save_sample_rows=False):
    model.eval()

    total_counts = {"tp": 0.0, "fp": 0.0, "fn": 0.0, "tn": 0.0}
    sample_rows = []

    for idx, batch in enumerate(tqdm(loader, desc=f"Threshold {threshold:.2f}", leave=False)):
        x = batch["x"].to(device)
        y_fire = batch["y_fire"].to(device)

        logits = model(x)

        counts = binary_segmentation_counts(logits.detach(), y_fire.detach(), threshold=threshold)
        total_counts = merge_counts(total_counts, counts)

        if save_sample_rows:
            metrics = metrics_from_counts(counts)
            pred_pixels = int((torch.sigmoid(logits) >= threshold).sum().item())
            target_pixels = int(y_fire.sum().item())

            sample_rows.append({
                "idx": idx,
                "fire_id": batch["fire_id"][0],
                "start_day": int(batch["start_day"][0]),
                "threshold": threshold,
                **counts,
                **metrics,
                "target_pixels": target_pixels,
                "pred_pixels": pred_pixels,
            })

    metrics = metrics_from_counts(total_counts)
    metrics.update(total_counts)
    metrics["threshold"] = threshold

    return metrics, sample_rows


thresholds = [
    0.05, 0.10, 0.15, 0.20, 0.25,
    0.30, 0.35, 0.40, 0.45, 0.50,
    0.55, 0.60, 0.65, 0.70, 0.75,
    0.80, 0.85, 0.90,
]

val_sweep_results = []

for th in thresholds:
    metrics, _ = evaluate_loader_at_threshold(
        model=eval_model,
        loader=full_val_loader,
        threshold=th,
        device=device,
        save_sample_rows=False,
    )

    val_sweep_results.append(metrics)

    print(
        f"Threshold {th:.2f} | "
        f"IoU: {metrics['iou']:.4f} | "
        f"F1: {metrics['f1']:.4f} | "
        f"Precision: {metrics['precision']:.4f} | "
        f"Recall: {metrics['recall']:.4f}"
    )

df_val_sweep = pd.DataFrame(val_sweep_results)
VAL_SWEEP_DIR = REPORT_DIR / "final_validation_sweep"
VAL_SWEEP_DIR.mkdir(parents=True, exist_ok=True)

val_sweep_csv = VAL_SWEEP_DIR / "threshold_sweep_val.csv"
df_val_sweep.to_csv(val_sweep_csv, index=False)

best_by_f1 = df_val_sweep.sort_values("f1", ascending=False).iloc[0]
best_by_iou = df_val_sweep.sort_values("iou", ascending=False).iloc[0]

summary = {
    "checkpoint": str(CHECKPOINT_PATH),
    "samples": len(full_val_ds),
    "best_by_f1": best_by_f1.to_dict(),
    "best_by_iou": best_by_iou.to_dict(),
    "note": "Threshold selected using validation split only.",
}

summary_json = VAL_SWEEP_DIR / "threshold_sweep_val_summary.json"
with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nSaved validation sweep:", val_sweep_csv)
print("Saved summary:", summary_json)

display(df_val_sweep)
print("\nBest by F1:")
display(best_by_f1)

## 14. Test Evaluation at Selected Threshold

The test split is evaluated using the threshold selected from validation.  
The default selected threshold is 0.65.

In [ ]:
test_ds = WildfireUNetNormalizedFireFilteredDataset(
    cache_root=CACHE_ROOT,
    split="test",
    normalization_stats_path=STATS_PATH,
    window_size=WINDOW_SIZE,
    crop_size=CROP_SIZE,
    crop_mode="fire_aware",
    positive_crop_prob=1.0,
    require_fire=True,
    min_fire_pixels=MIN_FIRE_PIXELS,
    limit_samples=None,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Filtered test samples:", len(test_ds))

test_metrics, test_sample_rows = evaluate_loader_at_threshold(
    model=eval_model,
    loader=test_loader,
    threshold=SELECTED_THRESHOLD,
    device=device,
    save_sample_rows=True,
)

print("\nTest metrics at threshold", SELECTED_THRESHOLD)
for key, value in test_metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value}")

TEST_EVAL_DIR = REPORT_DIR / "final_test_evaluation"
TEST_EVAL_DIR.mkdir(parents=True, exist_ok=True)

df_test_samples = pd.DataFrame(test_sample_rows)
test_sample_csv = TEST_EVAL_DIR / f"test_sample_metrics_threshold{SELECTED_THRESHOLD}.csv"
df_test_samples.to_csv(test_sample_csv, index=False)

test_summary_json = TEST_EVAL_DIR / f"test_summary_threshold{SELECTED_THRESHOLD}.json"
with open(test_summary_json, "w", encoding="utf-8") as f:
    json.dump(
        {
            "checkpoint": str(CHECKPOINT_PATH),
            "threshold": SELECTED_THRESHOLD,
            "samples": len(test_ds),
            "metrics": test_metrics,
            "note": "Filtered fire-aware patch-level test evaluation.",
        },
        f,
        indent=2,
    )

print("\nSaved sample metrics:", test_sample_csv)
print("Saved summary:", test_summary_json)

display(df_test_samples.head())

## 15. Prediction Visualization Utilities

Overlay color meaning:

```text
Yellow = correct overlap
Red    = false positive
Green  = missed fire
Black  = correct background
```

In [ ]:
def make_prediction_overlay(gt_fire, pred_mask):
    overlay = np.zeros((*gt_fire.shape, 3), dtype=np.float32)

    gt = gt_fire > 0.5
    pred = pred_mask > 0.5

    overlay[gt & ~pred] = [0, 1, 0]
    overlay[pred & ~gt] = [1, 0, 0]
    overlay[gt & pred] = [1, 1, 0]

    return overlay


def save_prediction_figure(x, y_fire, logits, output_path, threshold=0.65, title=""):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if x.dim() == 4:
        x = x[0]

    if y_fire.dim() == 4:
        y_fire = y_fire[0]

    if logits.dim() == 4:
        logits = logits[0]

    current_fire = x[30].detach().cpu().numpy()
    gt_fire = y_fire[0].detach().cpu().numpy()

    probs = torch.sigmoid(logits)[0].detach().cpu().numpy()
    pred_mask = (probs >= threshold).astype(np.float32)

    overlay = make_prediction_overlay(gt_fire, pred_mask)

    tp = int(((pred_mask == 1) & (gt_fire == 1)).sum())
    fp = int(((pred_mask == 1) & (gt_fire == 0)).sum())
    fn = int(((pred_mask == 0) & (gt_fire == 1)).sum())
    tn = int(((pred_mask == 0) & (gt_fire == 0)).sum())

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    iou = tp / max(tp + fp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-7)
    accuracy = (tp + tn) / max(tp + fp + fn + tn, 1)

    current_pixels = int((current_fire > 0.5).sum())
    target_pixels = int((gt_fire > 0.5).sum())
    pred_pixels = int(pred_mask.sum())

    plt.figure(figsize=(20, 5))

    plt.subplot(1, 4, 1)
    plt.imshow(current_fire, cmap="gray", vmin=0, vmax=1)
    plt.title(f"Input Fire Mask t\npixels={current_pixels}")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(gt_fire, cmap="gray", vmin=0, vmax=1)
    plt.title(f"Ground Truth t+1\npixels={target_pixels}")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(probs, cmap="hot", vmin=0, vmax=1)
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title(f"Predicted Probability\nmean={probs.mean():.3f}, max={probs.max():.3f}")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(overlay)
    plt.title(f"Overlay @ {threshold}\nIoU={iou:.3f}, F1={f1:.3f}")
    plt.axis("off")

    if title:
        plt.suptitle(title)

    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.show()
    plt.close()

    return {
        "current_fire_pixels": current_pixels,
        "target_fire_pixels": target_pixels,
        "pred_fire_pixels": pred_pixels,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
        "prob_min": float(probs.min()),
        "prob_max": float(probs.max()),
        "prob_mean": float(probs.mean()),
        "prob_median": float(np.median(probs)),
        "threshold": threshold,
        "output_path": str(output_path),
    }

## 16. Generate Test Prediction Visualizations

This cell saves prediction examples from the test set.

In [ ]:
PRED_DIR = REPORT_DIR / "final_prediction_examples_test_threshold065"
PRED_DIR.mkdir(parents=True, exist_ok=True)

NUM_EXAMPLES = 20
rows = []

eval_model.eval()

with torch.no_grad():
    for idx in range(min(NUM_EXAMPLES, len(test_ds))):
        sample = test_ds[idx]

        x = sample["x"].unsqueeze(0).to(device)
        y_fire = sample["y_fire"].unsqueeze(0).to(device)

        logits = eval_model(x)

        fire_id = sample["fire_id"]
        start_day = int(sample["start_day"])

        output_path = PRED_DIR / f"test_{idx:03d}_fire_{fire_id}_day_{start_day}_th065.png"

        stats = save_prediction_figure(
            x=x.cpu(),
            y_fire=y_fire.cpu(),
            logits=logits.cpu(),
            output_path=output_path,
            threshold=SELECTED_THRESHOLD,
            title=f"TEST | idx={idx}, fire={fire_id}, start_day={start_day}",
        )

        stats["idx"] = idx
        stats["fire_id"] = fire_id
        stats["start_day"] = start_day
        rows.append(stats)

        print("Saved:", output_path)

df_pred = pd.DataFrame(rows)
pred_csv = PRED_DIR / "prediction_stats.csv"
df_pred.to_csv(pred_csv, index=False)

print("\nSaved prediction stats:", pred_csv)
display(df_pred[[
    "idx", "fire_id", "start_day",
    "current_fire_pixels", "target_fire_pixels", "pred_fire_pixels",
    "iou", "precision", "recall", "f1"
]])

## 17. Final Output Summary

In [ ]:
print("Run directory:")
print(RUN_DIR)

print("\nSelected checkpoint:")
print(CHECKPOINT_PATH)

print("\nTraining log:")
print(LOG_DIR / "full_training_log.csv")

print("\nValidation sweep folder:")
print(VAL_SWEEP_DIR)

print("\nTest evaluation folder:")
print(TEST_EVAL_DIR)

print("\nPrediction examples folder:")
print(PRED_DIR)